# Perturb-JEPA Bridge: Colab End-to-End

This Colab notebook is a runnable start-to-final-evaluation workflow for the leakage-safe Perturb-JEPA Bridge repo.

It covers:

1. Clone the repository and install dependencies.
2. Set up GPU/paths and optional Google Drive persistence.
3. Download public data assets through the repo script.
4. Run the full staged pipeline on the built-in synthetic smoke data.
5. Run retrieval and counterfactual evaluations.
6. Save checkpoints and metrics artifacts.

The current stage scripts in the repo are synthetic-first scaffolds. They verify the full training/evaluation plumbing and produce checkpoints/metrics. Public data download and manifest cells are included so real-data wiring can be added without changing the notebook structure.


## 0. Runtime Notes

In Colab, choose `Runtime > Change runtime type > T4 GPU` or better.

Before running:

- Set `REPO_URL` to your GitHub repo URL/branch.
- If the repo is private, use a private-token clone URL or authenticate with GitHub first.
- Keep `RUN_SYNTHETIC_PIPELINE=True` for the first run. It should finish quickly and validates the whole workflow.
- Set `RUN_REAL_PIPELINE=True` only after `RNA_H5AD_PATH` and, for bridge training, `IMAGE_MANIFEST_PATH` point to real files.
- Bridge/fine-tune real training requires RNA and image metadata to share biological conditions: `perturbation`, `dose`, `time`, `cell_line`.
- Public scRNA files can be large. Start with `bf-moa-metadata` or a small h5ad subset before downloading everything.


In [ ]:
#@title 1. Configure repo, paths, and runtime
from pathlib import Path
import os
import shlex
import subprocess
import sys
import textwrap

REPO_URL = "https://github.com/yashraj59/perturb-jepa-bridge.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
PROJECT_DIR = Path("/content/perturb-jepa-bridge")  #@param {type:"string"}
DATA_DIR = Path("/content/perturb_jepa_data")  #@param {type:"string"}
OUTPUT_DIR = Path("/content/perturb_jepa_outputs")  #@param {type:"string"}
USE_GOOGLE_DRIVE = False  #@param {type:"boolean"}
RUN_SYNTHETIC_PIPELINE = True  #@param {type:"boolean"}
RUN_REAL_PIPELINE = False  #@param {type:"boolean"}

if USE_GOOGLE_DRIVE:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
    OUTPUT_DIR = Path('/content/drive/MyDrive/perturb_jepa_outputs')

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'checkpoints').mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'metrics').mkdir(parents=True, exist_ok=True)

DEVICE = 'cuda' if subprocess.run(['bash', '-lc', 'command -v nvidia-smi >/dev/null && nvidia-smi >/dev/null 2>&1']).returncode == 0 else 'cpu'
print(f'DEVICE={DEVICE}')
print(f'DATA_DIR={DATA_DIR}')
print(f'OUTPUT_DIR={OUTPUT_DIR}')


In [ ]:
#@title 2. Clone or update the repository
if PROJECT_DIR.exists():
    print(f"Repository already exists at {PROJECT_DIR}; fetching updates.")
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'fetch', '--all', '--prune'], check=True)
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only'], check=False)
else:
    clone_cmd = ['git', 'clone', '--branch', BRANCH, REPO_URL, str(PROJECT_DIR)]
    print(' '.join(shlex.quote(part) for part in clone_cmd))
    subprocess.run(clone_cmd, check=True)

os.chdir(PROJECT_DIR)
print('Current commit:')
subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], check=True)


In [ ]:
#@title 3. Install dependencies
# The data/dev extras install anndata, scipy, pillow, pytest, and PyYAML.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[data,dev]'], check=True)

import torch
print('torch', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))


## 4. Optional Public Data Download

The repo provides a conservative public-data downloader. It prints/runs explicit `curl` downloads for known public assets.

Suggested first downloads:

- `bf-moa-metadata`: small BF-MoA metadata archive.
- `bf-moa-manifest`: Figshare article manifest for image plate archives.
- One scPerturb h5ad, such as `srivatsan-sciplex3`, if you want RNA data locally.

Raw microscopy images are large; the raw-image cell is disabled by default and requires uncommenting selected plate IDs.


In [ ]:
#@title 4. Download selected public data assets
DOWNLOAD_PUBLIC_DATA = False  #@param {type:"boolean"}
DATASET_KEYS = "bf-moa-metadata,bf-moa-manifest"  #@param {type:"string"}
DRY_RUN = True  #@param {type:"boolean"}

if DOWNLOAD_PUBLIC_DATA:
    keys = [key.strip() for key in DATASET_KEYS.split(',') if key.strip()]
    for key in keys:
        cmd = [
            sys.executable,
            'scripts/download_public_data.py',
            '--dataset', key,
            '--output-dir', str(DATA_DIR / 'raw'),
        ]
        if DRY_RUN:
            cmd.append('--dry-run')
        print('\n$', ' '.join(shlex.quote(part) for part in cmd))
        subprocess.run(cmd, check=True)
else:
    print('Skipping data download. Set DOWNLOAD_PUBLIC_DATA=True when ready.')


In [ ]:
#@title 5. Build BF-MoA normalized manifest if metadata archive is present
bf_archive = DATA_DIR / 'raw' / 'bf_moa_data_tables.tar.gz'
bf_manifest = DATA_DIR / 'processed' / 'bf_moa_manifest.csv'

if bf_archive.exists():
    bf_manifest.parent.mkdir(parents=True, exist_ok=True)
    cmd = [
        sys.executable,
        'scripts/build_bf_moa_manifest.py',
        '--data-tables', str(bf_archive),
        '--output', str(bf_manifest),
        '--image-root', str(DATA_DIR / 'raw' / 'bf_moa_images'),
    ]
    print('$', ' '.join(shlex.quote(part) for part in cmd))
    subprocess.run(cmd, check=True)
    import pandas as pd
    manifest = pd.read_csv(bf_manifest)
    print(manifest.shape)
    display(manifest.head())
else:
    print(f'BF-MoA metadata archive not found at {bf_archive}. Run the previous cell with DOWNLOAD_PUBLIC_DATA=True and DRY_RUN=False.')


In [ ]:
#@title 5A. Download BF-MoA raw image plate archives (optional, large)
# Raw BF-MoA images are large. Keep this disabled until you have enough Colab/Drive disk.
DOWNLOAD_BF_MOA_RAW_IMAGES = False  #@param {type:"boolean"}
EXTRACT_BF_MOA_RAW_IMAGES = True  #@param {type:"boolean"}
KEEP_BF_MOA_ARCHIVES = False  #@param {type:"boolean"}
MAX_BF_MOA_ARCHIVES = 1  #@param {type:"integer"}

# Uncomment one or more plates. Start with one plate before downloading the full dataset.
BF_MOA_PLATES = """
# P015076
# P015077
# P015080
# P015081
# P015082
# P015083
# P015084
# P015085
# P015090
# P015091
# P015092
# P015093
# P015094
# P015095
# P015096
# P015097
# P015098
# P015099
""".strip()

raw_dir = DATA_DIR / 'raw'
bf_figshare_manifest = raw_dir / 'bf_moa_figshare_manifest.json'
bf_image_root = raw_dir / 'bf_moa_images'
bf_image_root.mkdir(parents=True, exist_ok=True)

def _selected_bf_moa_plates(text: str) -> list[str]:
    plates = []
    for line in text.splitlines():
        line = line.strip()
        if not line or line.startswith('#'):
            continue
        line = line.split('#', 1)[0].strip().strip(',')
        if line:
            plates.append(line)
    return plates

if DOWNLOAD_BF_MOA_RAW_IMAGES:
    if not bf_figshare_manifest.exists():
        cmd = [
            sys.executable,
            'scripts/download_public_data.py',
            '--dataset', 'bf-moa-manifest',
            '--output-dir', str(raw_dir),
        ]
        print('\n$', ' '.join(shlex.quote(str(part)) for part in cmd))
        subprocess.run(cmd, check=True)

    import json
    article = json.load(open(bf_figshare_manifest))
    files = article.get('files', article if isinstance(article, list) else [])
    requested_plates = _selected_bf_moa_plates(BF_MOA_PLATES)
    if not requested_plates:
        raise ValueError('No BF-MoA plates selected. Uncomment at least one plate in BF_MOA_PLATES.')

    archives = [
        file_info for file_info in files
        if any(plate in file_info.get('name', '') for plate in requested_plates)
        and file_info.get('name', '').lower().endswith(('.tar.gz', '.tgz', '.zip'))
    ]
    archives = sorted(archives, key=lambda item: item.get('name', ''))
    if MAX_BF_MOA_ARCHIVES and MAX_BF_MOA_ARCHIVES > 0:
        archives = archives[:MAX_BF_MOA_ARCHIVES]
    if not archives:
        raise ValueError(f'No Figshare archives matched selected plates: {requested_plates}')

    for file_info in archives:
        name = file_info['name']
        url = file_info.get('download_url') or file_info.get('downloadUrl')
        if not url:
            raise ValueError(f'Missing download URL for {name}')
        archive_path = raw_dir / name
        expected_size = int(file_info.get('size') or 0)
        if archive_path.exists() and (expected_size <= 0 or archive_path.stat().st_size == expected_size):
            print('Already downloaded', archive_path)
        else:
            print('Downloading', name, 'size_gb=', round(expected_size / 1024**3, 2))
            subprocess.run(['curl', '-L', url, '-o', str(archive_path)], check=True)
        if EXTRACT_BF_MOA_RAW_IMAGES:
            print('Extracting', archive_path, 'to', bf_image_root)
            lower_name = name.lower()
            if lower_name.endswith(('.tar.gz', '.tgz')):
                subprocess.run(['tar', '-xzf', str(archive_path), '-C', str(bf_image_root)], check=True)
            elif lower_name.endswith('.zip'):
                subprocess.run(['unzip', '-q', '-n', str(archive_path), '-d', str(bf_image_root)], check=True)
            else:
                print('Skipping extraction for unknown archive type:', name)
        if EXTRACT_BF_MOA_RAW_IMAGES and not KEEP_BF_MOA_ARCHIVES:
            archive_path.unlink(missing_ok=True)
    print('Image root exists:', bf_image_root.exists(), bf_image_root)
    subprocess.run(['bash', '-lc', f'find {shlex.quote(str(bf_image_root))} -type f | head'], check=False)
else:
    print('Skipping raw BF-MoA image downloads. Set DOWNLOAD_BF_MOA_RAW_IMAGES=True and uncomment selected BF_MOA_PLATES.')


## 6. Leakage-Safe Metadata Check

This checks that biological keys are separated from technical nuisance keys. Batch, plate, run, well, site, z-plane, sequencing lane, and library ID must not enter the biological condition key.


In [ ]:
#@title 6. Smoke-test the metadata schema
from perturb_jepa.data.schema import make_bio_key, make_tech_key, make_condition_id, validate_metadata_columns
import pandas as pd

example = pd.DataFrame([
    {
        'perturbation': 'drugA',
        'dose': '10uM',
        'time': '48h',
        'cell_line': 'U2OS',
        'batch': 'batch1',
        'plate': 'plate7',
        'well': 'A01',
        'site': '1',
        'sequencing_lane': 'L002',
        'library_id': 'lib42',
    }
])
validate_metadata_columns(example)
row = example.iloc[0].to_dict()
print('bio_key:', make_bio_key(row))
print('tech_key:', make_tech_key(row))
print('condition_id:', make_condition_id(row))


## 7. Real-Data Training Setup

Real-data training is now wired into the stage scripts.

Required for RNA pretraining and RNA counterfactual training:

- `RNA_H5AD_PATH`: AnnData `.h5ad` with `obs` columns `perturbation`, `perturbation_type`, `dose`, `time`, `cell_line`, `batch`.

Required for image pretraining:

- `IMAGE_MANIFEST_PATH`: CSV with image paths and normalized metadata columns.
- `IMAGE_ROOT`: directory containing image files referenced by the manifest.

Required for bridge/fine-tune real training:

- both RNA and image files must contain overlapping biological condition IDs.
- If no overlap exists, the bridge script fails clearly rather than creating fake pairs.


In [ ]:
#@title 7A. Real-data file paths and run sizes
RNA_H5AD_PATH = str(DATA_DIR / 'raw' / 'SrivatsanTrapnell2020_sciplex3.h5ad')  #@param {type:"string"}
IMAGE_MANIFEST_PATH = str(DATA_DIR / 'processed' / 'bf_moa_manifest.csv')  #@param {type:"string"}
IMAGE_ROOT = str(DATA_DIR / 'raw' / 'bf_moa_images')  #@param {type:"string"}

REAL_MAX_CELLS = 2048  #@param {type:"integer"}
REAL_MAX_IMAGES = 2048  #@param {type:"integer"}
REAL_TOP_GENES = 512  #@param {type:"integer"}
REAL_BATCH_SIZE = 4  #@param {type:"integer"}

from pathlib import Path
print('RNA exists:', Path(RNA_H5AD_PATH).exists(), RNA_H5AD_PATH)
print('Image manifest exists:', Path(IMAGE_MANIFEST_PATH).exists(), IMAGE_MANIFEST_PATH)
print('Image root exists:', Path(IMAGE_ROOT).exists(), IMAGE_ROOT)


## 7. Full Synthetic Training Pipeline

This runs all stages end-to-end using the repository's synthetic data path. It produces checkpoints and validates that the architecture, losses, stage scripts, and evaluations are wired correctly.

For a first Colab pass, use small step counts. Increase them only after the smoke run succeeds.


In [ ]:
#@title 7. Choose training step counts
PRETRAIN_RNA_STEPS = 2  #@param {type:"integer"}
PRETRAIN_IMAGE_STEPS = 2  #@param {type:"integer"}
BRIDGE_STEPS = 2  #@param {type:"integer"}
COUNTERFACTUAL_STEPS = 2  #@param {type:"integer"}
FINETUNE_STEPS = 2  #@param {type:"integer"}

def run(cmd: list[object]):
    cmd = [str(part) for part in cmd]
    print('\n$', ' '.join(shlex.quote(part) for part in cmd))
    subprocess.run(cmd, check=True)


In [ ]:
#@title 8. Stage 1A: RNA pretraining
if RUN_SYNTHETIC_PIPELINE:
    run([
        sys.executable,
        'scripts/train_pretrain_rna.py',
        '--config', 'configs/pretrain_rna.yaml',
        '--synthetic',
        '--steps', str(PRETRAIN_RNA_STEPS),
        '--device', DEVICE,
        '--checkpoint-out', str(OUTPUT_DIR / 'checkpoints' / 'pretrain_rna.pt'),
    ])
else:
    print('RUN_SYNTHETIC_PIPELINE=False; skipping synthetic RNA pretraining.')


In [ ]:
#@title 9. Stage 1B: Image pretraining
if RUN_SYNTHETIC_PIPELINE:
    run([
        sys.executable,
        'scripts/train_pretrain_image.py',
        '--config', 'configs/pretrain_image.yaml',
        '--synthetic',
        '--steps', str(PRETRAIN_IMAGE_STEPS),
        '--device', DEVICE,
        '--checkpoint-out', str(OUTPUT_DIR / 'checkpoints' / 'pretrain_image.pt'),
    ])
else:
    print('RUN_SYNTHETIC_PIPELINE=False; skipping synthetic image pretraining.')


In [ ]:
#@title 10. Stage 2: Bridge training
if RUN_SYNTHETIC_PIPELINE:
    run([
        sys.executable,
        'scripts/train_bridge.py',
        '--config', 'configs/bridge_train.yaml',
        '--synthetic',
        '--steps', str(BRIDGE_STEPS),
        '--device', DEVICE,
        '--checkpoint-out', str(OUTPUT_DIR / 'checkpoints' / 'bridge.pt'),
    ])
else:
    print('RUN_SYNTHETIC_PIPELINE=False; skipping synthetic bridge training.')


In [ ]:
#@title 11. Stage 3: Counterfactual training
if RUN_SYNTHETIC_PIPELINE:
    run([
        sys.executable,
        'scripts/train_counterfactual.py',
        '--config', 'configs/counterfactual_train.yaml',
        '--synthetic',
        '--steps', str(COUNTERFACTUAL_STEPS),
        '--device', DEVICE,
        '--checkpoint-out', str(OUTPUT_DIR / 'checkpoints' / 'counterfactual.pt'),
    ])
else:
    print('RUN_SYNTHETIC_PIPELINE=False; skipping synthetic counterfactual training.')


In [ ]:
#@title 12. Stage 4: Joint fine-tuning
if RUN_SYNTHETIC_PIPELINE:
    run([
        sys.executable,
        'scripts/train_finetune.py',
        '--config', 'configs/finetune.yaml',
        '--synthetic',
        '--steps', str(FINETUNE_STEPS),
        '--device', DEVICE,
        '--checkpoint-out', str(OUTPUT_DIR / 'checkpoints' / 'finetune.pt'),
    ])
else:
    print('RUN_SYNTHETIC_PIPELINE=False; skipping synthetic fine-tuning.')


## 8. Real-Data Training Pipeline

These cells run on real files when `RUN_REAL_PIPELINE=True`.

The bridge and fine-tune cells require overlapping biological conditions between RNA and image data. If your public RNA and BF-MoA image data do not overlap by compound/dose/time/cell line, run RNA/image pretraining separately and provide a matched manifest before bridge training.


In [ ]:
#@title 8A. Real RNA pretraining
if RUN_REAL_PIPELINE:
    run([
        sys.executable,
        'scripts/train_pretrain_rna.py',
        '--config', 'configs/pretrain_rna.yaml',
        '--rna-anndata', RNA_H5AD_PATH,
        '--max-cells', str(REAL_MAX_CELLS),
        '--n-top-genes', str(REAL_TOP_GENES),
        '--batch-size', str(REAL_BATCH_SIZE),
        '--steps', str(PRETRAIN_RNA_STEPS),
        '--device', DEVICE,
        '--checkpoint-out', str(OUTPUT_DIR / 'checkpoints' / 'real_pretrain_rna.pt'),
    ])
else:
    print('RUN_REAL_PIPELINE=False; skipping real RNA pretraining.')


In [ ]:
#@title 8B. Real image pretraining
if RUN_REAL_PIPELINE:
    run([
        sys.executable,
        'scripts/train_pretrain_image.py',
        '--config', 'configs/pretrain_image.yaml',
        '--image-manifest', IMAGE_MANIFEST_PATH,
        '--image-root', IMAGE_ROOT,
        '--max-images', str(REAL_MAX_IMAGES),
        '--batch-size', str(REAL_BATCH_SIZE),
        '--steps', str(PRETRAIN_IMAGE_STEPS),
        '--device', DEVICE,
        '--checkpoint-out', str(OUTPUT_DIR / 'checkpoints' / 'real_pretrain_image.pt'),
    ])
else:
    print('RUN_REAL_PIPELINE=False; skipping real image pretraining.')


In [ ]:
#@title 8C. Real bridge training
if RUN_REAL_PIPELINE:
    run([
        sys.executable,
        'scripts/train_bridge.py',
        '--config', 'configs/bridge_train.yaml',
        '--rna-anndata', RNA_H5AD_PATH,
        '--image-manifest', IMAGE_MANIFEST_PATH,
        '--image-root', IMAGE_ROOT,
        '--max-cells', str(REAL_MAX_CELLS),
        '--max-images', str(REAL_MAX_IMAGES),
        '--n-top-genes', str(REAL_TOP_GENES),
        '--batch-size', str(REAL_BATCH_SIZE),
        '--steps', str(BRIDGE_STEPS),
        '--device', DEVICE,
        '--checkpoint-out', str(OUTPUT_DIR / 'checkpoints' / 'real_bridge.pt'),
    ])
else:
    print('RUN_REAL_PIPELINE=False; skipping real bridge training.')


In [ ]:
#@title 8D. Real RNA counterfactual training
if RUN_REAL_PIPELINE:
    run([
        sys.executable,
        'scripts/train_counterfactual.py',
        '--config', 'configs/counterfactual_train.yaml',
        '--rna-anndata', RNA_H5AD_PATH,
        '--max-cells', str(REAL_MAX_CELLS),
        '--n-top-genes', str(REAL_TOP_GENES),
        '--batch-size', str(REAL_BATCH_SIZE),
        '--steps', str(COUNTERFACTUAL_STEPS),
        '--device', DEVICE,
        '--checkpoint-out', str(OUTPUT_DIR / 'checkpoints' / 'real_counterfactual.pt'),
    ])
else:
    print('RUN_REAL_PIPELINE=False; skipping real counterfactual training.')


In [ ]:
#@title 8E. Real fine-tuning
if RUN_REAL_PIPELINE:
    run([
        sys.executable,
        'scripts/train_finetune.py',
        '--config', 'configs/finetune.yaml',
        '--rna-anndata', RNA_H5AD_PATH,
        '--image-manifest', IMAGE_MANIFEST_PATH,
        '--image-root', IMAGE_ROOT,
        '--max-cells', str(REAL_MAX_CELLS),
        '--max-images', str(REAL_MAX_IMAGES),
        '--n-top-genes', str(REAL_TOP_GENES),
        '--batch-size', str(REAL_BATCH_SIZE),
        '--steps', str(FINETUNE_STEPS),
        '--device', DEVICE,
        '--checkpoint-out', str(OUTPUT_DIR / 'checkpoints' / 'real_finetune.pt'),
    ])
else:
    print('RUN_REAL_PIPELINE=False; skipping real fine-tuning.')


## 9. Final Evaluation

Retrieval evaluation reports learned embedding retrieval and leakage baselines. Counterfactual evaluation reports RNA response metrics and image latent/prototype metrics.


In [ ]:
#@title 13. Retrieval evaluation
retrieval_csv = OUTPUT_DIR / 'metrics' / 'retrieval_metrics.csv'
run([
    sys.executable,
    'scripts/evaluate_retrieval.py',
    '--synthetic',
    '--output', str(retrieval_csv),
])

import pandas as pd
retrieval_report = pd.read_csv(retrieval_csv)
display(retrieval_report)


In [ ]:
#@title 14. Counterfactual evaluation
counterfactual_csv = OUTPUT_DIR / 'metrics' / 'counterfactual_metrics.csv'
run([
    sys.executable,
    'scripts/evaluate_counterfactual.py',
    '--synthetic',
    '--output', str(counterfactual_csv),
])

counterfactual_report = pd.read_csv(counterfactual_csv)
display(counterfactual_report.head(30))


In [ ]:
#@title 14B. Real retrieval evaluation from bridge checkpoint
real_bridge_ckpt = OUTPUT_DIR / 'checkpoints' / 'real_bridge.pt'
real_retrieval_csv = OUTPUT_DIR / 'metrics' / 'real_retrieval_metrics.csv'
if RUN_REAL_PIPELINE and real_bridge_ckpt.exists():
    run([
        sys.executable,
        'scripts/evaluate_retrieval.py',
        '--checkpoint', str(real_bridge_ckpt),
        '--rna-anndata', RNA_H5AD_PATH,
        '--image-manifest', IMAGE_MANIFEST_PATH,
        '--image-root', IMAGE_ROOT,
        '--max-cells', str(REAL_MAX_CELLS),
        '--max-images', str(REAL_MAX_IMAGES),
        '--n-top-genes', str(REAL_TOP_GENES),
        '--batch-size', str(REAL_BATCH_SIZE),
        '--device', DEVICE,
        '--output', str(real_retrieval_csv),
        '--save-embeddings-dir', str(OUTPUT_DIR / 'embeddings'),
    ])
    display(pd.read_csv(real_retrieval_csv))
else:
    print('Skipping real retrieval evaluation; run real bridge training first.')


In [ ]:
#@title 14C. Real RNA counterfactual evaluation from checkpoint
real_cf_ckpt = OUTPUT_DIR / 'checkpoints' / 'real_counterfactual.pt'
real_cf_csv = OUTPUT_DIR / 'metrics' / 'real_counterfactual_metrics.csv'
if RUN_REAL_PIPELINE and real_cf_ckpt.exists():
    run([
        sys.executable,
        'scripts/evaluate_counterfactual.py',
        '--checkpoint', str(real_cf_ckpt),
        '--rna-anndata', RNA_H5AD_PATH,
        '--max-cells', str(REAL_MAX_CELLS),
        '--n-top-genes', str(REAL_TOP_GENES),
        '--device', DEVICE,
        '--output', str(real_cf_csv),
        '--save-arrays-dir', str(OUTPUT_DIR / 'counterfactual_arrays'),
    ])
    display(pd.read_csv(real_cf_csv).head(30))
else:
    print('Skipping real counterfactual evaluation; run real counterfactual training first.')


In [ ]:
#@title 15. Run the repo test suite as a final sanity check
RUN_TESTS = False  #@param {type:"boolean"}
if RUN_TESTS:
    run([sys.executable, '-m', 'pytest'])
else:
    print('Skipping tests. Set RUN_TESTS=True to run the full suite.')


## 10. Real-Data Notes

Real-data loaders are wired into the stage scripts. The bridge and fine-tune stages still require the RNA and image datasets to share biological conditions; this is a scientific requirement, not a software workaround.

If a downloaded scRNA dataset and BF-MoA image manifest do not overlap by `perturbation`, `dose`, `time`, and `cell_line`, use the RNA/image pretraining stages independently, then create or provide a matched condition manifest before bridge training.

The policy remains: encoders receive RNA/image tensors only; metadata is used for bag grouping, losses/evaluation labels, batch adversarial targets, and counterfactual conditioning.


In [ ]:
#@title 16. Save artifacts as a zip
archive_base = OUTPUT_DIR.parent / 'perturb_jepa_colab_artifacts'
archive_path = str(archive_base) + '.zip'
subprocess.run(['bash', '-lc', f'cd {shlex.quote(str(OUTPUT_DIR.parent))} && zip -qr {shlex.quote(Path(archive_path).name)} {shlex.quote(OUTPUT_DIR.name)}'], check=True)
print('Wrote', archive_path)

try:
    from google.colab import files  # type: ignore
    DOWNLOAD_ZIP = False  #@param {type:"boolean"}
    if DOWNLOAD_ZIP:
        files.download(archive_path)
except Exception:
    pass


## Done

You now have:

- staged training checkpoints under `OUTPUT_DIR/checkpoints`
- retrieval metrics under `OUTPUT_DIR/metrics/retrieval_metrics.csv`
- counterfactual metrics under `OUTPUT_DIR/metrics/counterfactual_metrics.csv`
- optional downloaded public data under `DATA_DIR`

Next step for real experiments: push the implementation to GitHub, update `REPO_URL`, and wire real AnnData/image manifests into the condition-bag datasets.
